# 01 - First circuit and load flow

## Objective

Build a source-line-load circuit and compare the load-bus voltage returned by direct OpenDSS with the same typed Case run through the CEPT public CLI.

## Source, assumptions, and units

The source is the checked-in `public/examples/first_circuit_case.json` definition: a three-phase, 12.47 kV source and load bus joined by `line1`, length 1.0 km, with the declared positive- and zero-sequence ohmic data. The load is 100.0 kW at power factor 0.95. These are demonstrator inputs, not measurements. Voltage magnitude is line-to-neutral per unit (`pu`); line length is km and load power is kW.

## Prediction

The load-bus voltage should be below the 1.0 pu source voltage because the feeder has nonzero impedance and the load consumes real and reactive power. Direct OpenDSS and CEPT should agree within a declared teaching tolerance because they use the same source assumptions.

## Action

First write the exact example input into this notebook's working directory, then solve it directly in OpenDSS. The CEPT action is a streamed subprocess call to `cept study run`; its output is not hidden behind a Python API.

## Verification

The CLI's exact run directory contains `case.json`, `results.json`, `manifest.json`, `validation_report.json`, and `public-verification.json`. `cept study verify` must return literal `passed: true`. The comparison uses solver-returned values from the direct calculation and `results.json`.

## Interpretation

The difference is a translation/regression check for this matched demonstrator. It is not a project acceptance tolerance and does not replace review of source data.

## Exercise

Change exactly one declared input in `CASE_PAYLOAD`, predict the direction of the voltage change, restart the kernel, and rerun every cell. Keep the changed input explicit; do not tune an output to meet the tolerance.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide the public CEPT CLI and OpenDSS runtime. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run first, then read the results below
import hashlib
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import urllib.parse
import urllib.request
from pathlib import Path
import html
from IPython.display import HTML, display

# Notebook workspace root, captured before any solver call: OpenDSS
# DataPath changes the process working directory, so later cells must not
# rely on Path.cwd().
WORKSPACE = Path.cwd()

DEFAULT_WHEEL_URL = 'https://github.com/sarutesri/cept-studio-edu/releases/download/v0.2.0-edu.1/cept_power_studio-0.2.0.dev0-py3-none-any.whl'
DEFAULT_WHEEL_SHA256 = 'c7e609a1d9cc85b322bfb615f0c796c7ea5c43815b197289eb555786964478bc'
configured_url = os.environ.get('CEPT_WHEEL_URL')
WHEEL_URL = (DEFAULT_WHEEL_URL if configured_url is None and importlib.util.find_spec('cept') is None else (configured_url or '')).strip()
WHEEL_SHA256 = os.environ.get('CEPT_WHEEL_SHA256', DEFAULT_WHEEL_SHA256 if WHEEL_URL == DEFAULT_WHEEL_URL else '').strip().lower()
if WHEEL_URL:
    if len(WHEEL_SHA256) != 64 or any(character not in '0123456789abcdef' for character in WHEEL_SHA256):
        raise ValueError('CEPT_WHEEL_SHA256 must be the caller-provided 64-character SHA-256')
    wheel_path = Path.cwd() / Path(urllib.parse.urlparse(WHEEL_URL).path).name
    print(f'Downloading caller-provided wheel: {WHEEL_URL}')
    urllib.request.urlretrieve(WHEEL_URL, wheel_path)
    digest = hashlib.sha256(wheel_path.read_bytes()).hexdigest()
    if digest != WHEEL_SHA256:
        raise ValueError(f'wheel hash mismatch: expected {WHEEL_SHA256}, got {digest}')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', str(wheel_path)], check=True)
else:
    print('CEPT_WHEEL_URL not supplied; using the existing installed environment.')
CLI_BIN = Path(sys.executable).parent / ('cept.exe' if os.name == 'nt' else 'cept')
if not CLI_BIN.is_file():
    raise RuntimeError('cept launcher not found next to Python; reinstall the pinned public wheel')
CLI = str(CLI_BIN)
print('CLI: cept --version')
print(subprocess.run([CLI, '--version'], capture_output=True, text=True, check=True).stdout.strip())
def run_cli(*arguments, verbose=False):
    # Quiet by default: result tables below are the lesson. Pass verbose=True
    # to stream the full solver-backed receipt instead.
    display_cmd = 'cept ' + shlex.join([str(argument) for argument in arguments])
    command = [CLI, *[str(argument) for argument in arguments]]
    print('$ ' + display_cmd, flush=True)
    if verbose:
        process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=Path.cwd())
        lines = []
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            lines.append(line)
        returncode = process.wait()
        output = ''.join(lines)
    else:
        completed = subprocess.run(command, capture_output=True, text=True, cwd=Path.cwd())
        returncode, output = completed.returncode, completed.stdout + completed.stderr
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, ['cept', *[str(argument) for argument in arguments]], output=output[-4000:])
    print('\u2192 exit 0', flush=True)
    return json.loads(output) if output.strip().startswith('{') else output

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def show_table(headers, rows):
    print('| ' + ' | '.join(headers) + ' |')
    print('| ' + ' | '.join('---' for _ in headers) + ' |')
    for row in rows:
        print('| ' + ' | '.join(str(value) for value in row) + ' |')

def cards(items, title='CEPT Studio'):
    blocks = []
    for label, value, note in items:
        blocks.append(f'''<div style="flex:1;min-width:180px;border:1px solid #d9dee8;border-radius:14px;padding:14px 16px;background:#fff;box-shadow:0 1px 3px rgba(0,0,0,.05)"><div style="font-size:12px;color:#667085;text-transform:uppercase;letter-spacing:.04em">{html.escape(str(label))}</div><div style="font-size:22px;font-weight:700;margin:4px 0;color:#182230">{html.escape(str(value))}</div><div style="font-size:12px;color:#667085">{html.escape(str(note))}</div></div>''')
    display(HTML(f'''<div style="font-family:Inter,Arial,sans-serif;margin:10px 0 18px"><div style="font-size:18px;font-weight:700;margin-bottom:9px">{html.escape(title)}</div><div style="display:flex;gap:10px;flex-wrap:wrap">{"".join(blocks)}</div></div>'''))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0


In [2]:
# @title Inputs \u2014 demonstrator values (no need to edit)
CASE_PATH = WORKSPACE / 'first_circuit_case.json'
CASE_PAYLOAD = {
    'meta': {'name': 'public_first_circuit', 'author': 'CEPT', 'description': 'Small three-phase source-line-load example used by the public first load-flow lesson.', 'mode': 'demonstrator'},
    'network': {
        'kind': 'inline', 'frequency_hz': 60,
        'inline': {
            'buses': [
                {'name': 'source', 'kv': 12.47, 'phases': 3},
                {'name': 'load', 'kv': 12.47, 'phases': 3},
            ],
            'lines': [{'name': 'line1', 'from_bus': 'source', 'to_bus': 'load', 'length_km': 1.0, 'r1_ohm_per_km': 0.2, 'x1_ohm_per_km': 0.4, 'r0_ohm_per_km': 0.6, 'x0_ohm_per_km': 1.2, 'b1_us_per_km': 0.0}],
            'loads': [{'id': 'load1', 'bus': 'load', 'phases': 3, 'kw': 100.0, 'pf': 0.95}],
            'external_grids': [{'name': 'grid', 'bus': 'source', 'pu': 1.0, 'angle_deg': 0.0, 'sk3_mva': 1000.0, 'x_r_ratio': 10.0}],
        },
    },
    'study': {'type': 'load_flow'},
}
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + '\n', encoding='utf-8')
print('Source input:', CASE_PATH)
show_table(['declared input', 'value', 'unit'], [('frequency', 60, 'Hz'), ('line length', 1.0, 'km'), ('load', 100.0, 'kW'), ('load power factor', 0.95, '1')])


Source input: <notebook-workspace>\first_circuit_case.json
| declared input | value | unit |
| --- | --- | --- |
| frequency | 60 | Hz |
| line length | 1.0 | km |
| load | 100.0 | kW |
| load power factor | 0.95 | 1 |


In [3]:
import opendssdirect as dss

for command in [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
assert dss.Solution.Converged()
dss.Circuit.SetActiveBus('load')
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
show_table(['source', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', phase, direct_by_phase[phase], 'pu') for phase in (1, 2, 3)])
assert all(value > 0 for value in direct_by_phase.values())


| source | phase | voltage magnitude | unit |
| --- | --- | --- | --- |
| direct OpenDSS | 1 | 0.9997586726902581 | pu |
| direct OpenDSS | 2 | 0.9997586726902756 | pu |
| direct OpenDSS | 3 | 0.9997586726902107 | pu |


In [4]:
RUN_DIR = WORKSPACE / 'runs' / '01-first-circuit'
run_summary = run_cli('study', 'run', CASE_PATH, '--out', RUN_DIR, '--force')
verify_summary = run_cli('study', 'verify', RUN_DIR)
results = read_json(RUN_DIR / 'results.json')
cept_rows = [row for row in results['load_flow']['bus_voltages'] if row['bus'].lower() == 'load']
cept_by_phase = {row['phase']: row['v_pu'] for row in cept_rows}
show_table(['source', 'phase', 'voltage magnitude', 'unit'], [('CEPT results.json', row['phase'], row['v_pu'], 'pu') for row in cept_rows])
show_table(['phase', 'direct OpenDSS pu', 'CEPT pu', 'absolute difference pu'], [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)])

max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Max |direct \u2212 CEPT|', f'{max_abs_diff_pu:.2e} pu', 'load-bus voltage agreement'),
], title='1 \u00b7 Load-flow agreement')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3)) < 1e-4


$ cept study run '<notebook-workspace>\first_circuit_case.json' --out '<notebook-workspace>\runs\01-first-circuit' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\01-first-circuit'


→ exit 0


| source | phase | voltage magnitude | unit |
| --- | --- | --- | --- |
| CEPT results.json | 1 | 0.999787 | pu |
| CEPT results.json | 2 | 0.999787 | pu |
| CEPT results.json | 3 | 0.999787 | pu |
| phase | direct OpenDSS pu | CEPT pu | absolute difference pu |
| --- | --- | --- | --- |
| 1 | 0.9997586726902581 | 0.999787 | 2.8327309741893458e-05 |
| 2 | 0.9997586726902756 | 0.999787 | 2.8327309724351935e-05 |
| 3 | 0.9997586726902107 | 0.999787 | 2.832730978929998e-05 |


The tables are generated from solver-returned direct values and the exact CEPT `results.json`; no output value is stored in the notebook. The verification receipt proves this public workflow's identity, convergence, finite quantities, and artifact integrity. It does not prove a real network or protection/project acceptance.